# Ayrton — EVCS no IEEE 8500

Eletropostos AC e hubs DC no alimentador **IEEE 8500** (redução de média tensão, ~11 × 12 km). Este notebook mostra os resultados já gerados e, se quiser, roda tudo de novo.

- Caso base **S0** (sem estações), **triagem** de locais e **S1** intradiário (15 min) para 2000–5000 carros.
- Hipóteses em `configs/evcs.yaml` (`planning:`). Ajustes da rede em relação aos dados IEEE: `data/ieee8500/README.md` (seção *Ajustes do estudo*).
- Documentação do módulo: `src/evcs/README.md`.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
from IPython.display import Image, display
CONFIGS = ROOT / 'configs'
RESULTS = ROOT / 'results' / 'ieee8500'
S0, SCREENING, S1 = RESULTS / 's0', RESULTS / 'evcs_screening', RESULTS / 's1'

## Rodar de novo (opcional)

Os resultados ficam em `results/` (fora do Git). Mude `RUN` para `True` para regenerar: S0 ~2 min, triagem ~1,5 h na primeira vez (a varredura de capacidade fica salva), S1 ~6 min por frota.

In [ ]:
RUN = False
if RUN:
    from integration.s0_study import export_s0
    from integration.evcs_screening import export_screening
    from integration.s1_study import export_s1
    export_s0(S0, CONFIGS, 'ieee8500')
    export_screening(SCREENING, CONFIGS)
    export_s1(S1, CONFIGS, SCREENING)

## Caso base S0

In [ ]:
s0 = json.loads((S0 / 'summary.json').read_text(encoding='utf-8'))
pd.Series({k: v for k, v in s0.items() if k != 'graph'})

In [ ]:
display(Image(S0 / 'topology.png', width=800))
display(Image(S0 / 'voltage_profile.png', width=1000))

## Triagem: capacidade da rede e alocação

In [ ]:
fleets = pd.read_csv(SCREENING / 'dados' / 'fleet_scenarios.csv').set_index('evs')
fleets[['hubs', 'eletropostos', 'chargers_dc', 'chargers_ac', 'sim_peak_15min_kw', 'served_share',
        'coverage_hub', 'coverage_eletroposto', 'network_accepts', 'binding', 'v_min_pu']].T

In [ ]:
display(Image(SCREENING / 'hosting_map.png', width=800))
display(Image(SCREENING / 'coverage_5000_evs.png', width=800))

## Curvas intradiárias (15 min)

Todas as frotas contra o caso base S0: demanda das estações, potência na subestação, tensão mínima e maior carregamento de linha.

In [ ]:
display(Image(S1 / 'intraday_sensitivity.png', width=1000))

In [ ]:
s0_day = pd.read_csv(S1 / 'dados' / 's0_intraday.csv')
table = {'S0': s0_day.agg({'p_kw': 'max', 'v_min_pu': 'min', 'line_max_pct': 'max', 'equipment_max_pct': 'max'})}
for d in sorted(S1.glob('evs_*'), key=lambda d: int(d.name.split('_')[1])):
    day = pd.read_csv(d / 'dados' / 'intraday.csv')
    table[d.name.replace('evs_', '') + ' carros'] = day.agg({'p_kw': 'max', 'v_min_pu': 'min',
                                                             'line_max_pct': 'max', 'equipment_max_pct': 'max'})
pd.DataFrame(table).rename(index={'p_kw': 'ponta subestação (kW)', 'v_min_pu': 'tensão mínima (pu)',
                                  'line_max_pct': 'maior carregamento de linha (%)',
                                  'equipment_max_pct': 'maior carregamento regulador/trafo (%)'}).round(3)

Detalhe de uma frota (mude `EVS`): demanda a 1 min / 15 min / 1 h e tensão a cada passo.

In [ ]:
EVS = 5000
for name in ('curves_intraday', 'voltage_intraday', 'delta_v_map'):
    display(Image(S1 / f'evs_{EVS}' / f'{name}.png', width=1000))